In [ ]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, asin, sqrt
import unicodedata

In [ ]:
# read datasets
acs = pd.read_csv("/kaggle/input/acs-2023-five-year/ACSDP5Y2023.DP03-Data.csv")
places = pd.read_csv("/kaggle/input/cdc-places-2024/PLACES__Local_Data_for_Better_Health_County_Data_2024_release_20250925.csv")
ahrf = pd.read_csv("/kaggle/input/ahrf-2023-2024/AHRF 2023-2024 CSV/ahrf2024_Feb2025.csv")
centroids = pd.read_csv("/kaggle/input/county-centroids/County Centroids.csv")
wonder = pd.read_csv("/kaggle/input/outcome-metrics/CDC_WONDER_2023_Updated.csv")

In [ ]:
# preprocess outcome data

us_states = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
    'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho',
    'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas',
    'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi',
    'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
    'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
    'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma',
    'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah',
    'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
    'WI': 'Wisconsin', 'WY': 'Wyoming', 'DC': 'District of Columbia'
}

wonder['County_raw'] = wonder['County'].str.strip()

valid_pattern = (
    r"^[A-Za-z\s'\.\-/]+ "
    r"(County|Parish|Borough|Census Area|Municipality|City and Borough|Borough/Census Area)"
    r", [A-Z]{2}$"
)

valid_mask = wonder['County_raw'].str.match(valid_pattern, na=False)
wonder_valid = wonder[valid_mask].copy()

wonder_valid[['County_raw', 'StateAbbr']] = wonder_valid['County_raw'].str.extract(
    r'^([A-Za-z\s\'\.\-/]+) (?:County|Parish|Borough|Census Area|Municipality|City and Borough|Borough/Census Area), ([A-Z]{2})$'
)
wonder_valid['State'] = wonder_valid['StateAbbr'].map(us_states)

wonder_valid['Deaths'] = wonder_valid['Deaths'].replace('Suppressed', '0')
wonder_valid['Deaths'] = pd.to_numeric(wonder_valid['Deaths'], errors='coerce')
wonder_valid = wonder_valid.dropna(subset=['Deaths'])

wonder_processed = wonder_valid.groupby(['County_raw', 'State'], as_index=False)['Deaths'].sum()
wonder_processed.rename(columns={'Deaths': 'diabetes_deaths_total'}, inplace=True)

In [ ]:
# filter to U.S. counties only
states_50 = list(us_states.values())

acs_filtered = acs[acs["NAME"].str.contains(",", na=False)].copy()
acs_filtered = acs_filtered[~acs_filtered["NAME"].str.contains("Puerto Rico", na=False)]
acs_filtered[["County_raw", "State"]] = acs_filtered["NAME"].str.split(",", n=1, expand=True)
acs_filtered["County_raw"] = acs_filtered["County_raw"].str.strip()
acs_filtered["State"] = acs_filtered["State"].str.strip()
acs_filtered = acs_filtered[acs_filtered["State"].isin(states_50)]

places_filtered = places.copy()
places_filtered['County_raw'] = places['LocationName'].str.strip()
places_filtered['State'] = places['StateDesc'].str.strip()
places_filtered = places_filtered[places_filtered["State"].isin(states_50)]

ahrf_filtered = ahrf.copy()
ahrf_filtered["County_raw"] = ahrf["cnty_name"].str.strip()
ahrf_filtered["State"] = ahrf["st_name"].str.strip()
ahrf_filtered = ahrf_filtered[ahrf_filtered["State"].isin(states_50)]

wonder_processed = wonder_processed[wonder_processed["State"].isin(states_50)]

In [ ]:
# pick acs columns

acs_column_mapping = {
    'State': 'State',
    'DP03_0009PE': 'unemployment_rate', 
    'DP03_0062E': 'median_household_income',
    'DP03_0088E': 'per_capita_income',
    'DP03_0096E': 'pop_with_health_insurance',
    'DP03_0099E': 'pop_without_health_insurance',
    'DP03_0098E': 'pop_with_public_insurance',
}

acs_selected = acs_filtered[[col for col in acs_column_mapping.keys() if col in acs_filtered.columns] + ['County_raw']].copy()
acs_selected.rename(columns=acs_column_mapping, inplace=True)

numeric_cols = ['unemployment_rate', 'median_household_income', 
                'per_capita_income',
                'pop_with_health_insurance', 'pop_without_health_insurance', 
                'pop_with_public_insurance']

for col in numeric_cols:
    acs_selected[col] = acs_selected[col].replace(['(X)', '-', 'N'], np.nan)
    acs_selected[col] = pd.to_numeric(acs_selected[col], errors='coerce')

In [ ]:
# pick ahrf columns

ahrf_column_mapping = {
    'State': 'State',
    'phys_nf_prim_care_pc_exc_rsdt_22': 'primary_care_physicians',
    'np_npi_23': 'nurse_practitioners',
    'pa_npi_23': 'physician_assistants',
    'hosp_beds_22': 'hospital_beds',
    'critcl_access_hosp_22': 'critical_access_hospitals',
    'stgh_diabetes_prevn_pgm_22': 'hospitals_diabetes_prevention',
    'stnglth_nutrtn_pgm_22': 'hospitals_nutrition_programs',
    'stgh_comn_hlth_eductn_22': 'hospitals_community_health_education',
    'popn_est_23': 'total_population',
}

ahrf_selected = ahrf_filtered[[col for col in ahrf_column_mapping.keys() if col in ahrf_filtered.columns] + ['County_raw']].copy()
ahrf_selected.rename(columns=ahrf_column_mapping, inplace=True)

In [ ]:
# pick places columns

places_key_measures = [
    'ACCESS2', 'BPHIGH', 'BPMED', 'CHD', 'CHECKUP', 'CHOLSCREEN',
    'CSMOKING', 'DIABETES', 'HIGHCHOL', 'LPA', 'OBESITY', 'STROKE',
]

places_filtered = places_filtered[places_filtered['MeasureId'].isin(places_key_measures)]

places_pivoted = places_filtered.pivot_table(
    index=['County_raw', 'State'],
    columns='MeasureId',
    values='Data_Value',
    aggfunc='first'
).reset_index()

places_pivoted.columns.name = None
places_pivoted.rename(columns={
    'ACCESS2': 'pct_no_health_insurance_places',
    'BPHIGH': 'pct_high_blood_pressure',
    'BPMED': 'pct_on_bp_medication',
    'CHD': 'pct_coronary_heart_disease',
    'CHECKUP': 'pct_annual_checkup',
    'CHOLSCREEN': 'pct_cholesterol_screening',
    'CSMOKING': 'pct_current_smoking',
    'DIABETES': 'pct_diabetes',
    'HIGHCHOL': 'pct_high_cholesterol',
    'LPA': 'pct_low_physical_activity',
    'OBESITY': 'pct_obesity',
    'STROKE': 'pct_stroke',
}, inplace=True)

In [ ]:
# standardize names

def strip_accents(text):
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )

def normalize_county(name, state):
    if pd.isna(name):
        return name
    
    n = strip_accents(name).strip()
    
    if state == "Alaska":
        n = (n.lower()
             .replace(" city and borough", "")
             .replace(" borough/census area", "")
             .replace(" city and", "")
             .replace(" city", "")
             .replace(" borough", "")
             .replace(" census area", "")
             .replace(" municipality", "")
             .replace("(b)", "")
             .replace("(ca)", "")
             .replace("-", " "))
        return n.title()
    
    if state == "Connecticut":
        n = n.replace(" Planning Region", "").replace(" Pr", "").strip().title()
        ct_mapping = {
            "Fairfield": "Western Connecticut",
            "New Haven": "Naugatuck Valley",
            "Middlesex": "Lower Connecticut River Valley",
            "Hartford": "Capitol",
            "Litchfield": "Northwest Hills",
            "New London": "Southeastern Connecticut",
            "Tolland": "Northeastern Connecticut",
            "Windham": "Northeastern Connecticut"
        }
        return ct_mapping.get(n, n)
    
    if state == "Louisiana":
        return n.replace(" Parish", "").replace("La Salle", "Lasalle").title()
    
    if state == "Virginia":
        return n.replace(" County", "").replace(" City", "").replace(" city", "").title()
    
    return n.replace(" County", "").strip().title()

acs_selected['County'] = acs_selected.apply(
    lambda r: normalize_county(r['County_raw'], r['State']), axis=1
)
ahrf_selected['County'] = ahrf_selected.apply(
    lambda r: normalize_county(r['County_raw'], r['State']), axis=1
)
places_pivoted['County'] = places_pivoted.apply(
    lambda r: normalize_county(r['County_raw'], r['State']), axis=1
)
wonder_processed['County'] = wonder_processed.apply(
    lambda r: normalize_county(r['County_raw'], r['State']), axis=1
)

ahrf_county_fixes = {
    ("Lasalle", "Illinois"): "Lasalle",
    ("Lasalle", "Texas"): "La Salle",
    ("La Porte", "Indiana"): "Laporte",
    ("Dekalb", "Indiana"): "Dekalb",
    ("De Kalb", "Indiana"): "Dekalb",
    ("Laporte", "Indiana"): "Laporte",
    ("Prince Of Wales-Outer Ketchikan", "Alaska"): "Prince Of Wales Hyder",
    ("Capitol Pr", "Connecticut"): "Capitol",
}

def fix_ahrf_county(row):
    key = (row['County'], row['State'])
    return ahrf_county_fixes.get(key, row['County'])

ahrf_selected['County'] = ahrf_selected.apply(fix_ahrf_county, axis=1)

places_pivoted['County'] = places_pivoted['County'].str.replace('Ã‚Â±', 'n', regex=False)

ahrf_selected = ahrf_selected[~ahrf_selected.set_index(['County', 'State']).index.isin([
    ("Valdez Cordova", "Alaska"),
    ("Clifton Forge", "Virginia")
])]

ahrf_selected = ahrf_selected.drop_duplicates(subset=['County', 'State'], keep='first')

acs_selected = acs_selected.drop(columns=['County_raw'])
ahrf_selected = ahrf_selected.drop(columns=['County_raw'])
places_pivoted = places_pivoted.drop(columns=['County_raw'])
wonder_processed = wonder_processed.drop(columns=['County_raw'])

In [ ]:
# merge all datasets

acs_keys = set(zip(acs_selected['County'], acs_selected['State']))
ahrf_keys = set(zip(ahrf_selected['County'], ahrf_selected['State']))
places_keys = set(zip(places_pivoted['County'], places_pivoted['State']))
wonder_keys = set(zip(wonder_processed['County'], wonder_processed['State']))

common_keys = acs_keys & ahrf_keys & places_keys & wonder_keys

print(f"Counties in ACS: {len(acs_keys)}")
print(f"Counties in AHRF: {len(ahrf_keys)}")
print(f"Counties in PLACES: {len(places_keys)}")
print(f"Counties in WONDER: {len(wonder_keys)}")
print(f"Counties in all datasets: {len(common_keys)}")

acs_selected = acs_selected[acs_selected.apply(
    lambda r: (r['County'], r['State']) in common_keys, axis=1
)]
ahrf_selected = ahrf_selected[ahrf_selected.apply(
    lambda r: (r['County'], r['State']) in common_keys, axis=1
)]
places_pivoted = places_pivoted[places_pivoted.apply(
    lambda r: (r['County'], r['State']) in common_keys, axis=1
)]
wonder_processed = wonder_processed[wonder_processed.apply(
    lambda r: (r['County'], r['State']) in common_keys, axis=1
)]

merged_df = (acs_selected
             .merge(places_pivoted, on=['County', 'State'], how='inner')
             .merge(ahrf_selected, on=['County', 'State'], how='inner')
             .merge(wonder_processed, on=['County', 'State'], how='inner'))

In [ ]:
# calculate insurance percentage and outcome

merged_df['pct_with_insurance'] = (merged_df['pop_with_health_insurance'] / 
                                    merged_df['total_population'] * 100)
merged_df['pct_without_insurance'] = (merged_df['pop_without_health_insurance'] / 
                                       merged_df['total_population'] * 100)
merged_df['pct_with_public_insurance'] = (merged_df['pop_with_public_insurance'] / 
                                           merged_df['total_population'] * 100)

merged_df['deaths_with_underlying_cause_diabetes_per_1k'] = (
    merged_df['diabetes_deaths_total'] / merged_df['total_population'] * 1000
)

In [ ]:
# spatial weighting

from scipy.spatial.distance import cdist

centroids['County'] = centroids['county'].str.replace(' County', '', regex=False).str.strip()
centroids['State'] = centroids['state'].str.strip()

centroids['County'] = centroids.apply(
    lambda r: normalize_county(r['County'], r['State']), axis=1
)

merged_with_coords = merged_df.merge(
    centroids[['County', 'State', 'latitude', 'longitude']],
    on=['County', 'State'],
    how='inner'
)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2 * R * asin(sqrt(a))

neighbor_weight_vars = [
    'primary_care_physicians',
    'nurse_practitioners',
    'physician_assistants',
    'hospital_beds',
    'critical_access_hospitals',
    'hospitals_diabetes_prevention',
    'hospitals_nutrition_programs',
    'hospitals_community_health_education'
]

lambda_decay = 0.05
max_distance = 100

coords = merged_with_coords[['latitude', 'longitude']].to_numpy()    
n = len(merged_with_coords)

dists = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        dists[i, j] = haversine(coords[i, 0], coords[i, 1], coords[j, 0], coords[j, 1])

weights_matrix = np.exp(-lambda_decay * (dists / 10))
weights_matrix[dists > max_distance] = 0

for var in neighbor_weight_vars:
    values = merged_with_coords[var].values
    weighted_vals = weights_matrix @ values
    merged_with_coords[f'{var}_per_1k'] = (weighted_vals / merged_with_coords['total_population'] * 1000)

In [ ]:
# drop unnecessary cols
columns_to_drop = [
    'latitude', 'longitude', 'total_population',
    'pop_with_health_insurance', 'pop_without_health_insurance', 'pop_with_public_insurance',
    'diabetes_deaths_total',
    'primary_care_physicians', 'nurse_practitioners', 'physician_assistants',
    'hospital_beds', 'critical_access_hospitals',
    'hospitals_diabetes_prevention', 'hospitals_nutrition_programs', 
    'hospitals_community_health_education'
]

final_df = merged_with_coords.drop(columns=columns_to_drop)

final_df.to_csv("full_diabetes_dataset.csv", index=False)
print(f"\nFinal dataset shape: {final_df.shape}")
print(f"\nTarget variable summary:")
print(final_df['deaths_with_underlying_cause_diabetes_per_1k'].describe())
final_df.head()